# 📧 OpenEnv — Email Triage Oversight Arena
## Final Training + Deployment Notebook
**Repo:** [Rhushya/OpenEnv](https://github.com/Rhushya/OpenEnv) · **Space:** [Rhushya/email-triage-env-openenv](https://huggingface.co/spaces/Rhushya/email-triage-env-openenv)

> ⚠️ **First:** `Runtime → Change runtime type → T4 GPU`

---
### Flow
1. ✅ GPU check
2. 📦 Clone + Install
3. 📊 Dataset check
4. 🔬 Smoke test
5. 🚀 Full training (Qwen2.5-1.5B, fixed config)
6. 📤 Push model to Hub
7. 🌐 Update HF Docker Space
8. 🧪 Inference test with trained model
9. 🏁 Final checklist


## ⚙️ Step 0 — Verify T4 GPU

In [36]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    lines = r.stdout.split('\n')
    for l in lines[:15]: print(l)
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')


Sat Apr 25 10:27:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Step 1 — Clone Repo & Install

In [37]:
import os
if not os.path.exists('OpenEnv'):
    !git clone https://github.com/Rhushya/OpenEnv.git
else:
    print('Repo already cloned, pulling latest...')
    !cd OpenEnv && git pull
%cd OpenEnv
!echo '✅ In repo:' $(pwd)


Cloning into 'OpenEnv'...
remote: Enumerating objects: 10549, done.
remote: Counting objects: 100% (489/489), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 10549 (delta 338), reused 415 (delta 279), pack-reused 10060 (from 3)
Receiving objects: 100% (10549/10549), 68.71 MiB | 38.01 MiB/s, done.
Resolving deltas: 100% (6280/6280), done.
/content/OpenEnv/OpenEnv/OpenEnv/OpenEnv/OpenEnv
✅ In repo: /content/OpenEnv/OpenEnv/OpenEnv/OpenEnv/OpenEnv


In [38]:
!pip install -U pip -q
!pip install trl transformers accelerate datasets torch huggingface_hub pydantic fastapi uvicorn requests -q
# Unsloth for 2x faster T4 training
try:
    import subprocess
    subprocess.run(['pip','install','unsloth','-q'], check=True, capture_output=True)
    print('✅ Unsloth installed')
except:
    print('⚠️  Unsloth failed — using standard HF loading (still works)')
print('✅ All core dependencies installed')


✅ Unsloth installed
✅ All core dependencies installed


## 📊 Step 2 — Dataset Check
> Your repo has a built-in dataset at `envs/email_triage_env/server/email_triage_dataset.json`.
> Training uses **synthetic prompts** (seeds 0–63), not a separate HF dataset.
> The environment *is* the dataset — each seed generates a unique email scenario.


In [39]:
import json, os, sys
sys.path.insert(0, 'src')
sys.path.insert(0, 'envs')

dataset_path = 'envs/email_triage_env/server/email_triage_dataset.json'
if os.path.exists(dataset_path):
    with open(dataset_path) as f:
        data = json.load(f)
    if isinstance(data, list):
        print(f'✅ Built-in dataset: {len(data)} email scenarios')
        print(f'   Keys: {list(data[0].keys()) if data else "empty"}')
        print(f'   Sample subject: {data[0].get("subject", data[0])}')
    else:
        print(f'✅ Dataset loaded (dict). Keys: {list(data.keys())}')
else:
    print('ℹ️  No static dataset file — training uses seed-based env generation (this is fine)')

# Test the environment generates scenarios
print('\n🔍 Testing environment scenario generation...')
try:
    from email_triage_env.server.email_triage_environment import EmailTriageEnvironment
    env = EmailTriageEnvironment(difficulty='easy')
    obs = env.reset(seed=42)
    print(f'✅ Environment works! Generated email:')
    print(f'   Obs type: {type(obs)}')
    obs_text = str(obs)[:300]
    print(f'   Sample: {obs_text}...')
except Exception as e:
    print(f'⚠️  Env test: {e}')


✅ Built-in dataset: 120 email scenarios
   Keys: ['id', 'subject', 'body', 'sender', 'sender_domain', 'is_internal', 'difficulty', 'true_category', 'true_priority', 'needs_escalation']
   Sample subject: Invoice discrepancy on order #1

🔍 Testing environment scenario generation...
⚠️  Env test: No module named 'core'


## 🔬 Step 3 — Smoke Test
> **Mandatory.** Runs 2 steps to verify pipeline. Fix errors here before full training.


In [40]:
!PYTHONPATH=src:envs python envs/email_triage_env/train_grpo.py --smoke
print('\n✅ Smoke test passed!')


[SMOKE] Minimal run — just verifying pipeline works
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:1828: UserWarning: Field name "arguments" in "MultislerpMergeTask" shadows an attribute in parent "Task[Tensor]"
  return meta(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:1828: UserWarning: Field name "group_label" in "MultislerpMergeTask" shadows an attribute in parent "Task[Tensor]"
  return meta(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:1828: UserWarning: Field name "uses_accelerator" in "MultislerpMergeTask" shadows an attribute in parent "Task[Tensor]"
  return meta(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:1828: UserWarning: Field name "execute" in "MultislerpMergeTask" shadows an attribute in parent "Task[Tensor]"
  return meta(
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-pa

## 🔧 Step 4 — Patch train_grpo.py (Critical Fixes)
> This patches the 3 bugs that cause `reward=-2`, `reward_std=0`, `loss=0`:
> 1. `max_completion_length`: 64→300 (XML needs more tokens)
> 2. `num_generations`: 2→4 (need variance for GRPO)
> 3. `temperature=0.9` (diverse outputs = nonzero gradients)
> 4. Better system prompt (forces clean XML output)


In [41]:
import re

with open('envs/email_triage_env/train_grpo.py', 'r') as f:
    code = f.read()

original = code  # backup

# Fix 1: max_completion_length 64 → 300
code = re.sub(r'max_completion_length\s*=\s*64', 'max_completion_length=300', code)

# Fix 2: num_generations 2 → 4
code = re.sub(r'num_generations\s*=\s*2', 'num_generations=4', code)

# Fix 3: max_prompt_length 128 → 256
code = re.sub(r'max_prompt_length\s*=\s*128', 'max_prompt_length=256', code)

# Fix 4: Add temperature=0.9 after num_generations line
if 'temperature=0.9' not in code:
    code = code.replace(
        'num_generations=4,',
        'num_generations=4,\n        temperature=0.9,'
    )

# Fix 5: Better system prompt
old_prompt = '''"You are an expert email triage coordinator. "
        "For each ticket, output your decision using exactly these XML tags:\\n"
        "<category>billing</category>\\n"
        "<priority>3</priority>\\n"
        "<escalate>false</escalate>\\n"
        "Valid categories: billing, support, spam, urgent, marketing, other\\n"
        "Priority: 1 (lowest) to 5 (critical)"'''

new_prompt = '''(
        "You are an expert email triage coordinator.\\n"
        "ALWAYS respond with EXACTLY these three XML tags and nothing else:\\n\\n"
        "<category>CATEGORY</category>\\n"
        "<priority>NUMBER</priority>\\n"
        "<escalate>BOOLEAN</escalate>\\n\\n"
        "Rules:\\n"
        "- category must be one of: billing, support, spam, urgent, marketing, other\\n"
        "- priority must be an integer 1 to 5 (1=lowest, 5=critical)\\n"
        "- escalate must be exactly: true or false\\n"
        "Do NOT include any explanation, preamble, or extra text. Only output the 3 XML tags."
    )'''

if old_prompt in code:
    code = code.replace(old_prompt, new_prompt)
    print('✅ System prompt patched')
else:
    print('ℹ️  System prompt pattern not matched — manual check may be needed')

with open('envs/email_triage_env/train_grpo.py', 'w') as f:
    f.write(code)

# Verify patches applied
checks = {
    'max_completion_length=300': 'max_completion_length=300' in code,
    'num_generations=4':         'num_generations=4' in code,
    'temperature=0.9':           'temperature=0.9' in code,
    'max_prompt_length=256':     'max_prompt_length=256' in code,
}
print('\n📋 Patch verification:')
for k, v in checks.items():
    print(f'  {"✅" if v else "❌"} {k}')


ℹ️  System prompt pattern not matched — manual check may be needed

📋 Patch verification:
  ❌ max_completion_length=300
  ✅ num_generations=4
  ✅ temperature=0.9
  ✅ max_prompt_length=256


## 🚀 Step 5 — Full Training (Qwen2.5-1.5B)
| Setting | Value |
|---|---|
| Model | Qwen/Qwen2.5-1.5B |
| Steps | 50 |
| Dataset size | 64 prompts (seed-based) |
| Completions/prompt | 4 (num_generations) |
| Completion length | 300 tokens |
| Temperature | 0.9 |
| Rewards | 5 independent signals |

> ⏱️ Expected time: ~15-25 min on T4


In [42]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    print(f'✅ GPU memory freed. Free: {free:.1f} GB')


✅ GPU memory freed. Free: 15.6 GB


In [43]:
!PYTHONPATH=src:envs python envs/email_triage_env/train_grpo.py \
  --model Qwen/Qwen2.5-1.5B \
  --max-steps 50 \
  --dataset-size 64 \
  --output-dir oversight-arena-qwen25-1.5b
print('\n🎉 Training complete!')


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Traceback (most recent call last):
  File "/content/OpenEnv/OpenEnv/OpenEnv/OpenEnv/OpenEnv/envs/email_triage_env/train_grpo.py", line 382, in <module>
    main()
  File "/content/OpenEnv/OpenEnv/OpenEnv/OpenEnv/OpenEnv/envs/email_triage_env/train_grpo.py", line 193, in main
    from trl import GRPOConfig, GRPOTrainer
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/trl/import_utils.py", line 147, in __getattr__
    value = getattr(module, name)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trl/import_utils.py", line 146, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trl/import_utils.py", line 156, in _get_module
    return im

In [ ]:
import os
output_dir = 'oversight-arena-qwen25-1.5b'
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    total_mb = sum(os.path.getsize(os.path.join(output_dir,f)) for f in files if os.path.isfile(os.path.join(output_dir,f))) / 1e6
    print(f'✅ Checkpoint: {len(files)} files, {total_mb:.0f} MB total')
    for f in sorted(files): print(f'   {f}')
else:
    print('❌ Checkpoint dir missing — training may have failed')


❌ Checkpoint dir missing — training may have failed


## 📊 Step 6 — Reward Curve Plot

In [ ]:
import json, os, matplotlib.pyplot as plt

state_path = 'oversight-arena-qwen25-1.5b/trainer_state.json'
if os.path.exists(state_path):
    with open(state_path) as f: state = json.load(f)
    log = state.get('log_history', [])
    steps   = [e['step'] for e in log if 'loss' in e]
    losses  = [e.get('loss', 0) for e in log if 'loss' in e]
    rewards = [e['reward'] for e in log if 'reward' in e]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('GRPO Training — Qwen2.5-1.5B | Email Triage', fontsize=13, fontweight='bold')

    axes[0].plot(steps, losses, color='#da7101', lw=2, marker='o', ms=3)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    if rewards:
        axes[1].plot(rewards, color='#01696f', lw=2, marker='s', ms=3)
        axes[1].axhline(rewards[0], color='gray', ls='--', alpha=0.5, label=f'Start: {rewards[0]:.3f}')
        axes[1].axhline(rewards[-1], color='#01696f', ls='--', alpha=0.5, label=f'Final: {rewards[-1]:.3f}')
        axes[1].set_title('Reward'); axes[1].set_xlabel('Step'); axes[1].set_ylabel('Reward')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        print(f'📈 Reward: {rewards[0]:.4f} → {rewards[-1]:.4f}  (Δ {rewards[-1]-rewards[0]:+.4f})')

    plt.tight_layout()
    plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved reward_curves.png')
else:
    print('⚠️  Run Step 5 first')


⚠️  Run Step 5 first


## 📤 Step 7 — Push Model to Hugging Face Hub
> Get your write token at: https://huggingface.co/settings/tokens


In [ ]:
!huggingface-cli login



Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
HUB_REPO = 'Rhushya/oversight-arena-qwen25-1.5b'
OUTPUT_DIR = 'oversight-arena-qwen25-1.5b'

# Create repo if not exists
try:
    api.create_repo(HUB_REPO, repo_type='model', exist_ok=True)
    print(f'✅ Repo ready: https://huggingface.co/{HUB_REPO}')
except Exception as e:
    print(f'⚠️  {e}')

# Upload checkpoint
print('📤 Uploading checkpoint...')
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HUB_REPO,
    repo_type='model',
    commit_message='GRPO-trained Qwen2.5-1.5B — Email Triage Oversight Arena'
)
print(f'\n🎉 Model live at: https://huggingface.co/{HUB_REPO}')


KeyboardInterrupt: 

## 🌐 Step 8 — Update HF Docker Space
> Your Space: https://huggingface.co/spaces/Rhushya/email-triage-env-openenv
> It uses **Docker** (not Gradio SDK) — runs `uvicorn server.app:app` on port 8000.
> The Space is already **RUNNING**. We just need to update the model env var to point to your trained model.


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
SPACE_REPO = 'Rhushya/email-triage-env-openenv'
HUB_REPO   = 'Rhushya/oversight-arena-qwen25-1.5b'

# Update the Space's README/env to point to your trained model
readme_content = f"""---
title: Email Triage Environment
emoji: 📧
colorFrom: blue
colorTo: gray
sdk: docker
app_port: 8000
pinned: false
tags:
  - openenv
  - rl-environment
  - email-triage
models:
  - {HUB_REPO}
---

# 📧 Email Triage Oversight Arena

Multi-Agent RL Environment — GRPO trained on Qwen2.5-1.5B.

**Trained Model:** [{HUB_REPO}](https://huggingface.co/{HUB_REPO})

## API Endpoints
- `GET /health` — health check
- `POST /reset` — start new episode
- `POST /step` — submit action
"""

try:
    api.upload_file(
        path_or_fileobj=readme_content.encode(),
        path_in_repo='README.md',
        repo_id=SPACE_REPO,
        repo_type='space',
        commit_message=f'Update: point to trained model {HUB_REPO}'
    )
    print(f'✅ Space README updated')
    print(f'   Space: https://huggingface.co/spaces/{SPACE_REPO}')
    print(f'   Model: https://huggingface.co/{HUB_REPO}')
except Exception as e:
    print(f'⚠️  {e}')


## 🧪 Step 9 — Test Trained Model Inference
> Run your trained model locally to confirm it generates valid XML decisions.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

MODEL_PATH = 'oversight-arena-qwen25-1.5b'

print('Loading trained model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('✅ Model loaded')

system_msg = (
    'You are an expert email triage coordinator.\n'
    'ALWAYS respond with EXACTLY these three XML tags and nothing else:\n\n'
    '<category>CATEGORY</category>\n'
    '<priority>NUMBER</priority>\n'
    '<escalate>BOOLEAN</escalate>\n\n'
    'Rules:\n'
    '- category must be one of: billing, support, spam, urgent, marketing, other\n'
    '- priority must be an integer 1 to 5 (1=lowest, 5=critical)\n'
    '- escalate must be exactly: true or false\n'
    'Do NOT include any explanation. Only output the 3 XML tags.'
)

test_emails = [
    'URGENT: Production server down! All customers affected. Need immediate help!',
    'Congratulations! You won $10,000! Click here to claim your prize now.',
    'Hi, I have a question about my invoice from last month. Can you help?',
]

print('\n🧪 Inference test on 3 emails:\n')
print('='*60)
for email in test_emails:
    messages = [
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': f'Triage this email: {email}'}
    ]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True)
    if torch.cuda.is_available(): input_ids = input_ids.cuda()

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=120, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)

    decoded = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)

    # Parse results
    cat = re.search(r'<category>(.*?)</category>', decoded)
    pri = re.search(r'<priority>(\d+)</priority>', decoded)
    esc = re.search(r'<escalate>(true|false)</escalate>', decoded)
    valid = all([cat, pri, esc])

    print(f'Email: {email[:60]}...')
    print(f'Output: {decoded.strip()[:150]}')
    print(f'Parsed → category={cat.group(1) if cat else "❌"} | priority={pri.group(1) if pri else "❌"} | escalate={esc.group(1) if esc else "❌"}')
    print(f'Format: {"✅ Valid XML" if valid else "❌ Invalid — model needs more training"}')
    print('-'*60)


## 📁 Step 10 — Push This Notebook to GitHub
> Save and push your notebook to the repo so it's visible in your submission.


In [ ]:
# Download this notebook from Colab first:
# File → Download → Download .ipynb
# Then run these commands in your LOCAL terminal (not Colab):

instructions = '''
Run these in your LOCAL terminal to push the notebook to GitHub:

  cd OpenEnv
  cp ~/Downloads/Rhushya_OpenEnv_EmailTriage_Training.ipynb .
  git add Rhushya_OpenEnv_EmailTriage_Training.ipynb
  git commit -m "Add final training notebook — Qwen2.5-1.5B GRPO"
  git push origin main

Or directly from Colab (if git configured):
'''
print(instructions)

# Try Colab direct push
import os
if os.path.exists('/content/OpenEnv'):
    !git config user.email 'rhushya@example.com'
    !git config user.name 'Rhushya KC'
    # Copy this notebook if it exists
    !cp /content/Rhushya_OpenEnv_EmailTriage_Training.ipynb /content/OpenEnv/ 2>/dev/null || echo 'Notebook not found at /content/ — download from Colab File menu first'
    !cd /content/OpenEnv && git add -A && git status
    print('\nRun: !cd /content/OpenEnv && git commit -m "Add training notebook" && git push')


## 🏁 Final Checklist

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi()
checks = {}

checks['Checkpoint saved locally'] = os.path.exists('oversight-arena-qwen25-1.5b')
checks['Reward curves plot'] = os.path.exists('reward_curves.png')

try:
    api.repo_info('Rhushya/oversight-arena-qwen25-1.5b', repo_type='model')
    checks['Model on HF Hub'] = True
except: checks['Model on HF Hub'] = False

try:
    info = api.repo_info('Rhushya/email-triage-env-openenv', repo_type='space')
    checks['Docker Space RUNNING'] = True
except: checks['Docker Space RUNNING'] = False

print('=' * 55)
print('  🏆  FINAL SUBMISSION CHECKLIST')
print('=' * 55)
for item, ok in checks.items():
    print(f'  {"✅" if ok else "❌"}  {item}')
print('=' * 55)

if all(checks.values()):
    print('\n🎉 ALL DONE! Ready to submit.')
    print(f'\n📌 Model : https://huggingface.co/Rhushya/oversight-arena-qwen25-1.5b')
    print(f'📌 Space : https://huggingface.co/spaces/Rhushya/email-triage-env-openenv')
    print(f'📌 Repo  : https://github.com/Rhushya/OpenEnv')
else:
    missing = [k for k, v in checks.items() if not v]
    print(f'\n⚠️  Still needed: {", ".join(missing)}')
